In [ ]:
import hashlib
import re
import shutil
import subprocess
import time
from pathlib import Path

import pandas as pd
from Bio import SeqIO

In [ ]:
MUESTREO_XLSX = "muestreo_PAIRED_89.xlsx"
RUN = pd.read_excel(MUESTREO_XLSX, sheet_name="Muestra_PAIRED_89")["run_accession"].iloc[0]

THREADS          = 12
FRACCION_SECCION = 0.10
N_SECCIONES      = 2
SEED             = 100

CARD_URL             = "https://card.mcmaster.ca/latest/data"
AMRFINDERPLUS_DB_DIR = Path("localDB/amrfinderplus")
AMRFINDER_DB         = AMRFINDERPLUS_DB_DIR / "latest"

RAW_DIR, WORK_DIR, OUT_DIR = Path(f"raw/{RUN}"), Path(f"work/{RUN}"), Path(f"results/{RUN}")
for d in (RAW_DIR, WORK_DIR, OUT_DIR):
    d.mkdir(parents=True, exist_ok=True)

print(f"Muestra: {RUN}")

In [ ]:
def sh(cmd):
    t0 = time.time()
    proc = subprocess.Popen(cmd, shell=True, text=True, bufsize=1,
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    for line in proc.stdout:
        el = time.time() - t0
        print(f"[{int(el // 60):>2}m{int(el % 60):02d}s] {line}", end="", flush=True)
    proc.wait()
    if proc.returncode != 0:
        raise RuntimeError(f"Fallo (codigo {proc.returncode}): {cmd}")


def md5sum(path, chunk=1 << 20):
    h = hashlib.md5()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(chunk), b""):
            h.update(block)
    return h.hexdigest()


def reintentar(fn, intentos=5, espera_s=5, descripcion="operacion"):
    ultimo_error = None
    for intento in range(1, intentos + 1):
        try:
            return fn()
        except Exception as e:
            ultimo_error = e
            if intento < intentos:
                print(f"  Fallo en {descripcion} ({e}); reintentando en {espera_s}s [{intento}/{intentos}]...")
                time.sleep(espera_s)
    raise RuntimeError(f"'{descripcion}' fallo tras {intentos} intentos: {ultimo_error}") from ultimo_error


def _consultar_ena_filereport(url, descripcion):
    def _get():
        df = pd.read_csv(url, sep="\t")
        if df.empty or "fastq_ftp" not in df.columns or pd.isna(df.loc[0, "fastq_ftp"]):
            raise ValueError(f"respuesta de ENA sin fastq_ftp utilizable: {df.to_dict('records')[:1]}")
        return df
    return reintentar(_get, descripcion=descripcion)


def descargar_con_aria2c(pares_url_nombre, dest_dir, threads):
    input_file = dest_dir / "aria2_input.txt"
    with open(input_file, "w") as f:
        for url, nombre in pares_url_nombre:
            f.write(f"{url}\n  out={nombre}\n")
    conexiones = min(threads, 16)
    sh(f"aria2c -i {input_file} -d {dest_dir} -x{conexiones} -s{conexiones} -j2 -c "
       f"--max-tries=0 --retry-wait=5 --timeout=120 --allow-overwrite=true --quiet=true")

In [ ]:
if not Path("localDB/card.json").exists():
    if not Path("card.json").exists():
        sh(f"wget -q {CARD_URL} -O card_data.tar.bz2")
        sh("tar -xjf card_data.tar.bz2 ./card.json")
    sh("rgi load --card_json card.json --local")
assert Path("localDB/card.json").exists(), "CARD no quedo cargada"

if not AMRFINDER_DB.exists():
    AMRFINDERPLUS_DB_DIR.mkdir(parents=True, exist_ok=True)
    sh(f"amrfinder_update -d {AMRFINDERPLUS_DB_DIR}")
assert AMRFINDER_DB.exists(), "AMRFinderPlus DB no quedo lista"

In [ ]:
ena_url = (
    "https://www.ebi.ac.uk/ena/portal/api/filereport"
    f"?accession={RUN}&result=read_run&fields=fastq_ftp,fastq_md5&format=tsv"
)
ena = _consultar_ena_filereport(ena_url, descripcion=f"consulta ENA ({RUN})")
ftp = ena.loc[0, "fastq_ftp"].split(";")
md5_esperado = ena.loc[0, "fastq_md5"].split(";")

r1, r2 = RAW_DIR / f"{RUN}_1.fastq.gz", RAW_DIR / f"{RUN}_2.fastq.gz"
if not all(r.exists() and md5sum(r) == m for r, m in zip((r1, r2), md5_esperado)):
    descargar_con_aria2c([("https://" + ftp[0], r1.name), ("https://" + ftp[1], r2.name)], RAW_DIR, THREADS)
    for archivo, esperado in zip((r1, r2), md5_esperado):
        obtenido = md5sum(archivo)
        if obtenido != esperado:
            raise RuntimeError(f"MD5 no coincide para {archivo.name}: {obtenido} != {esperado}")
print(f"Descarga verificada (MD5): {r1.name}, {r2.name}")

In [ ]:
clean1, clean2 = WORK_DIR / "clean_1.fastq.gz", WORK_DIR / "clean_2.fastq.gz"
sh(f"fastp -i {r1} -I {r2} -o {clean1} -O {clean2} -w {THREADS} "
   f"-h {OUT_DIR}/{RUN}_fastp.html -j {OUT_DIR}/{RUN}_fastp.json")

In [ ]:
LINEAS_POR_LECTURA = 4
frac_total = N_SECCIONES * FRACCION_SECCION
pool1, pool2 = WORK_DIR / "pool_1.fastq", WORK_DIR / "pool_2.fastq"

sh(f"seqtk sample -s{SEED} {clean1} {frac_total} > {pool1}")
sh(f"seqtk sample -s{SEED} {clean2} {frac_total} > {pool2}")

pool_lineas = int(subprocess.run(f"wc -l < {pool1}", shell=True, text=True, capture_output=True).stdout)
n_seccion = pool_lineas // LINEAS_POR_LECTURA // N_SECCIONES

sub_pares = {}
for i in range(1, N_SECCIONES + 1):
    ini_linea, n_lineas = (i - 1) * n_seccion * LINEAS_POR_LECTURA, n_seccion * LINEAS_POR_LECTURA
    sub1, sub2 = WORK_DIR / f"sub_1_{i}.fastq", WORK_DIR / f"sub_2_{i}.fastq"
    sh(f"tail -n +{ini_linea + 1} {pool1} | head -n {n_lineas} > {sub1}")
    sh(f"tail -n +{ini_linea + 1} {pool2} | head -n {n_lineas} > {sub2}")
    sub_pares[i] = (sub1, sub2)
    print(f"Seccion {i}: {n_seccion:,} pares")

pool1.unlink()
pool2.unlink()

In [ ]:
contigs, genes_faa, genes_fna, genes_gff = {}, {}, {}, {}
for i, (sub1, sub2) in sub_pares.items():
    megahit_out = WORK_DIR / f"megahit_out_{i}"
    shutil.rmtree(megahit_out, ignore_errors=True)
    sh(f"megahit -1 {sub1} -2 {sub2} -t {THREADS} -o {megahit_out}")
    contigs[i] = megahit_out / "final.contigs.fa"

    genes_faa[i], genes_fna[i], genes_gff[i] = (WORK_DIR / f"genes_{i}.faa", WORK_DIR / f"genes_{i}.fna",
                                                WORK_DIR / f"genes_{i}.gff")
    sh(f"prodigal -i {contigs[i]} -a {genes_faa[i]} -d {genes_fna[i]} -p meta -q -o {genes_gff[i]} -f gff")

In [ ]:
rgi_df, amrfinder_df = {}, {}
for i in genes_faa:
    rgi_out = OUT_DIR / f"rgi_{RUN}_{i}"
    sh(f"rgi main -i {genes_faa[i]} -o {rgi_out} -t protein -a DIAMOND --local --clean")
    rgi_df[i] = pd.read_csv(f"{rgi_out}.txt", sep="\t")

    amr_out = OUT_DIR / f"amrfinder_{RUN}_{i}.tsv"
    sh(f"amrfinder -p {genes_faa[i]} -n {contigs[i]} -g {genes_gff[i]} -a prodigal "
       f"-d {AMRFINDER_DB} -o {amr_out} --threads {THREADS}")
    amrfinder_df[i] = pd.read_csv(amr_out, sep="\t")

    print(f"Seccion {i}: {len(rgi_df[i])} ARG (CARD/RGI), {len(amrfinder_df[i])} ARG (AMRFinderPlus)")

In [ ]:
def normalizar_gen(nombre):
    return re.sub(r"[^A-Z0-9]", "", str(nombre).upper())


def _partes_gen(nombre):
    n = normalizar_gen(nombre)
    m = re.match(r"^([A-Z]*)([0-9].*)?$", n)
    return m.group(1), m.group(2) or ""


def mismo_gen(a, b, min_len=3):
    pa, na = _partes_gen(a)
    pb, nb = _partes_gen(b)
    if not pa or not pb:
        return False
    if na != nb:
        return False
    if len(pa) < min_len or len(pb) < min_len:
        return pa == pb
    return pa in pb or pb in pa


def cruzar_card_amrfinder(rgi, amrfinder):
    genes_card = sorted(rgi["Best_Hit_ARO"].dropna().unique())
    genes_amr = sorted(amrfinder["Element symbol"].dropna().unique())

    filas, vistos_amr = [], set()
    for g_card in genes_card:
        candidatos = [g for g in genes_amr if mismo_gen(g_card, g)]
        vistos_amr.update(candidatos)
        filas.append({"gen": g_card, "en_CARD": True, "en_AMRFinderPlus": bool(candidatos),
                      "gen_AMRFinderPlus": ", ".join(candidatos) or None})
    for g_amr in genes_amr:
        if g_amr not in vistos_amr:
            filas.append({"gen": g_amr, "en_CARD": False, "en_AMRFinderPlus": True, "gen_AMRFinderPlus": g_amr})
    cruce_gen = pd.DataFrame(filas, columns=["gen", "en_CARD", "en_AMRFinderPlus", "gen_AMRFinderPlus"])

    familias_card = sorted(rgi["Drug Class"].dropna().str.split(";").explode().str.strip().unique())
    clases_amr = sorted(amrfinder["Class"].dropna().unique())

    filas_fam, vistas_amr = [], set()
    for f_card in familias_card:
        candidatas = [c for c in clases_amr if mismo_gen(f_card, c)]
        vistas_amr.update(candidatas)
        filas_fam.append({"familia": f_card, "en_CARD": True, "en_AMRFinderPlus": bool(candidatas)})
    for c_amr in clases_amr:
        if c_amr not in vistas_amr:
            filas_fam.append({"familia": c_amr, "en_CARD": False, "en_AMRFinderPlus": True})
    cruce_familia = pd.DataFrame(filas_fam, columns=["familia", "en_CARD", "en_AMRFinderPlus"])

    return cruce_gen, cruce_familia


def contig_de_orf(orf, asm_ids):
    orf = str(orf).split()[0]
    if orf in asm_ids:
        return orf
    partes = orf.split("_")
    for corte in range(len(partes) - 1, 0, -1):
        cand = "_".join(partes[:corte])
        if cand in asm_ids:
            return cand
    return None


def coords_desde_header_prodigal(orf_id):
    m = re.search(r"#\s*(\d+)\s*#\s*(\d+)\s*#\s*(-?1)\s*#", str(orf_id))
    if not m:
        return pd.Series([None, None, None])
    inicio, fin, hebra = m.groups()
    return pd.Series([int(inicio), int(fin), int(hebra)])


def agregar_posicion_relativa(df, contigs_path, col_contig, col_start, col_stop, resolver_contig=None):
    largos = {r.id: len(r.seq) for r in SeqIO.parse(str(contigs_path), "fasta")}
    df = df.copy()
    if resolver_contig:
        asm_ids = set(largos)
        df["contig_asm"] = df[col_contig].map(lambda x: resolver_contig(x, asm_ids))
    else:
        df["contig_asm"] = df[col_contig]
    df["contig_len"] = df["contig_asm"].map(largos)
    df["pos_inicio_rel"] = df[col_start] / df["contig_len"]
    df["pos_fin_rel"] = df[col_stop] / df["contig_len"]
    return df


def agregar_secuencias(df, faa_path, fna_path, col_id):
    proteinas = {r.id: str(r.seq) for r in SeqIO.parse(str(faa_path), "fasta")}
    nucleotidos = {r.id: str(r.seq) for r in SeqIO.parse(str(fna_path), "fasta")}
    df = df.copy()
    ids = df[col_id].map(lambda x: str(x).split()[0])
    df["secuencia_aa"] = ids.map(proteinas)
    df["secuencia_nt"] = ids.map(nucleotidos)
    return df

In [ ]:
secciones = []
for i in genes_faa:
    cruce_gen, cruce_familia = cruzar_card_amrfinder(rgi_df[i], amrfinder_df[i])
    cruce_gen.to_csv(OUT_DIR / f"cruce_gen_{RUN}_{i}.csv", index=False)
    cruce_familia.to_csv(OUT_DIR / f"cruce_familia_{RUN}_{i}.csv", index=False)

    rgi = rgi_df[i].copy()
    if len(rgi):
        # rgi main -t protein deja Start/Stop/Orientation vacios -- se sacan del header de prodigal en ORF_ID
        rgi[["Start", "Stop", "Orientation"]] = rgi["ORF_ID"].apply(coords_desde_header_prodigal)
    rgi = agregar_posicion_relativa(rgi, contigs[i], "ORF_ID", "Start", "Stop", resolver_contig=contig_de_orf)
    amr = agregar_posicion_relativa(amrfinder_df[i], contigs[i], "Contig id", "Start", "Stop")

    rgi = agregar_secuencias(rgi, genes_faa[i], genes_fna[i], "ORF_ID")
    amr = agregar_secuencias(amr, genes_faa[i], genes_fna[i], "Protein id")

    rgi.insert(0, "herramienta", "CARD/RGI")
    amr.insert(0, "herramienta", "AMRFinderPlus")
    seccion = pd.concat([rgi, amr], ignore_index=True)
    seccion.insert(0, "seccion", i)
    secciones.append(seccion)

    ambos = (cruce_gen["en_CARD"] & cruce_gen["en_AMRFinderPlus"]).sum()
    solo_card = (cruce_gen["en_CARD"] & ~cruce_gen["en_AMRFinderPlus"]).sum()
    solo_amr = (~cruce_gen["en_CARD"] & cruce_gen["en_AMRFinderPlus"]).sum()
    print(f"Seccion {i} -- genes: {ambos} en ambas, {solo_card} solo CARD, {solo_amr} solo AMRFinderPlus")

resistoma = pd.concat(secciones, ignore_index=True)
resistoma.insert(0, "run", RUN)
resistoma.to_csv(OUT_DIR / f"resistoma_{RUN}.csv", index=False)

resistoma[["seccion", "herramienta", "Best_Hit_ARO", "Element symbol", "Drug Class", "Class",
           "contig_asm", "Start", "Stop", "contig_len", "pos_inicio_rel", "pos_fin_rel", "secuencia_aa"]]